# Week 14 — Monday: Fairness

**DATA 202 · Calvin University**

In the 1930s, a federal agency colored maps of U.S. cities by mortgage risk. Neighborhoods where Black families lived were outlined in red, "hazardous", and mortgages there became hard to get: **redlining**. Today the Fair Housing Act and the Equal Credit Opportunity Act forbid deciding a loan by race, and the Home Mortgage Disclosure Act (1975) makes lenders publish their decisions, so anyone can check.

In Week 11, a logistic regression learned from Kent County's 2025 decisions. It never saw the applicants' race, sex or age, and its test set has been locked since.

> 🎯 **Is Week 11's model fair? And fair by what measure?**

**Today's plan (~40 min):**

| Time | Part |
|---|---|
| ~5 min | 1 · Unlock the test set (SLO 14A) |
| ~10 min | 2 · Same threshold, two groups (SLO 14A) |
| ~8 min | 3 · Leave race out? (SLO 14A) |
| ~10 min | 4 · Fair by which measure? (SLO 14A) |
| ~7 min | 5 · A toolbox for fairness: which criterion, when? (SLO 14A) |

**Cues:** 🎯 Predict First · 💬 Question · 🗣️ Pair Talk · 📌 One split

<sub>Data: Home Mortgage Disclosure Act (HMDA) records for 2025, Consumer Financial Protection Bureau (public domain): applications for the applicant's own single-family home in Kent County, Michigan</sub>

---
## 1 · Unlock the Test Set (SLO 14A) · ~5 min

Week 11's model, exactly: the same six columns, the same split, the same pipeline. Week 11 chose it with cross-validation on the training applications. Today the test applications come out.

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.preprocessing import StandardScaler, OneHotEncoder

mortgages = pd.read_csv("https://cs.calvin.edu/courses/data/202/26fa/datasets/mortgages.csv")
numeric = ["debt_to_income", "loan_to_value", "loan_amount"]
words = ["loan_purpose", "lien", "loan_type"]
complete = mortgages.dropna(subset=numeric)

X = complete[numeric + words]                                   # no race, sex or age, as in Week 11
y = complete["decision"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=1, stratify=y)

model = Pipeline([
    ("prep", ColumnTransformer([
        ("num", StandardScaler(), numeric),
        ("words", OneHotEncoder(drop="first"), words),
    ])),
    ("logit", LogisticRegression(max_iter=5000)),
])
model.fit(X_train, y_train)
print(f"test accuracy: {model.score(X_test, y_test):.3f} | approve everyone: {(y_test == 'approved').mean():.3f}")

**88.2%** on the 4,429 test applications, close to Week 11's cross-validation (88.4%). Overall, the model holds up.

Say a lender uses it, with Week 11's threshold of **0.2**: the model denies every application with a chance of denial of at least 20%. (In Week 11's table, 0.2 caught about two-thirds of the lenders' denials.)

HMDA records each applicant's race, as the applicant reports it. The two largest groups are White and Black applicants.

> 🎯 **Predict First:** The model never sees race. Will it deny Black and White applicants at the same rate?

In [ ]:
test = complete.loc[X_test.index].copy()                        # the test applications, with every column
test["chance"] = model.predict_proba(X_test)[:, 1]              # the model's P(denied)
test["model_denies"] = test["chance"] >= 0.2                    # the lender's threshold
test["lender_denied"] = test["decision"] == "denied"            # what the lender really decided

two = test[test["race"].isin(["White", "Black or African American"])]
print(two["race"].value_counts())
two.groupby("race")[["lender_denied", "model_denies"]].mean().round(3)

<details><summary>Answer</summary>

* **The lenders** denied **30.7%** of Black applicants and **12.9%** of White ones
* **The model** denies **44.1%** of Black applicants and **26.0%** of White ones, without ever seeing race
* Is that unfair? It depends on what we compare. Part 2 compares four ways
</details>

---
## 2 · Same Threshold, Two Groups (SLO 14A) · ~10 min

Start small: twelve test applicants from each group.

> 📄 **Handout 1 — Same threshold, two groups.** Draw the threshold at 0.2 on both strips, and count. 🗣️ Which row would a Black applicant point to? Which would the lender?

In [ ]:
white12 = [1166, 2793, 6044, 12130, 8471, 9831, 9654, 14343, 18503, 17154, 3224, 1416]    # Handout 1's applicants
black12 = [8059, 9852, 15346, 1126, 3041, 2636, 6, 12204, 14654, 2525, 2393, 14674]
strips = test.loc[white12 + black12, ["race", "chance", "decision"]]
fig = px.strip(strips, x="chance", y="race", color="decision", stripmode="overlay", range_x=[0, 0.8],
               color_discrete_map={"approved": "#8c8c8c", "denied": "#eb6834"},
               labels={"chance": "the model's chance of denial", "race": "", "decision": "the lender:"},
               title="Twelve Test Applicants from Each Group")
fig.add_vline(x=0.2, line_dash="dash", annotation_text="threshold 0.2")
fig.show()

<details><summary>Answer</summary>

| | White | Black |
|---|---|---|
| the model denies | 3 of 12 | 5 of 12 |
| **wrongly denied:** of the applicants the lender approved, the model denies | 2 of 10 | 3 of 8 |
| **missed:** of the applicants the lender denied, the model approves | 1 of 2 | 2 of 4 |

* An approved Black applicant is turned away more often (3 of 8) than an approved White one (2 of 10): the applicant's complaint
* The lender's risk, approving someone it would have denied, is the same: half of each group's denials are missed
</details>

Twelve applicants are a handful. The same counts for every test applicant in the two groups, at every threshold:

In [ ]:
rows = []
for race, group in two.groupby("race"):
    approved, denied = ~group["lender_denied"], group["lender_denied"]
    for threshold in np.arange(0.05, 0.61, 0.01).round(2):
        denies = group["chance"] >= threshold
        rows.append({"race": race, "threshold": threshold,
                     "model denies": denies.mean(),
                     "wrongly denied": (denies & approved).sum() / approved.sum(),   # of the lender's approvals
                     "missed": (~denies & denied).sum() / denied.sum(),              # of the lender's denials
                     "precision": (denies & denied).sum() / denies.sum(),            # of the model's denials
                     "accuracy": (denies == denied).mean()})
measures = pd.DataFrame(rows)
measures[measures["threshold"] == 0.2].set_index("race").drop(columns="threshold").T.round(3)

> 💬 **Question:** Which gaps are big, and which are small? Who is each gap unfair to?

<details><summary>Answer</summary>

Each measure, made equal across groups, is a **fairness criterion** with a name:

* **Model denies:** 44% against 26%. Equal shares would be **demographic parity** ✗
* **Wrongly denied:** the model turns away **35%** of the Black applicants the lender approved, against **20%** of White ones. Equal shares would be **equal opportunity**: a qualified applicant gets the same chance in every group ✗. This is the applicant's complaint
* **Missed:** 35% and 34%. The lender's risk is the same for both groups. Equal *wrongly denied* and equal *missed* together would be **equalized odds**: half there
* **Precision:** 46% against 33%. When the model denies a Black applicant, the lender agrees more often. Equal precision would be **predictive parity** ✗, the other way around
* **Accuracy:** 65% against 78% ✗
* "Wrongly" means "disagrees with the lender". The lenders' decisions are the answer key. Part 3 asks whether the key itself is fair
</details>

> 📌 **One split:** 254 Black and 3,393 White test applicants. With 78 Black denials, one applicant moves *missed* by more than a point.

---
## 3 · Leave Race Out? (SLO 14A) · ~8 min

Leaving the protected columns out is called **fairness through unawareness**. For lenders, the law requires it. Yet the model denies Black applicants more often. Its columns differ by group:

* `tract_minority_pct`: the share of the residents of the applicant's census tract (a neighborhood of about 4,000 people) who belong to a racial or ethnic minority

In [ ]:
profile = two.assign(fha=two["loan_type"] == "FHA", cash_out=two["loan_purpose"] == "cash-out refinance")
profile.groupby("race")[["debt_to_income", "loan_to_value", "fha", "cash_out", "tract_minority_pct"]].mean().round(2)

<details><summary>Answer</summary>

* Black applicants in the test set: debt-to-income **42%** against 37%; **15%** FHA loans against 6%; **22%** cash-out refinances against 14%. Week 11's slopes push all three toward denial
* They also live in neighborhoods that are **47%** minority on average, against 23%
</details>

An experiment no lender may run: give the model the **race** column. Or, instead, the **neighborhood's** minority share, a column that never says "race".

> 🎯 **Predict First:** What happens to the gap between the groups' chances of denial?

In [ ]:
versions = {
    "Week 11 (no race)": numeric + words,
    "+ the neighborhood": numeric + words + ["tract_minority_pct"],
    "+ race (illegal for a lender)": numeric + words + ["race"],
}
black, white = two["race"] == "Black or African American", two["race"] == "White"
rows = {}
for name, columns in versions.items():
    version = Pipeline([
        ("prep", ColumnTransformer([
            ("num", StandardScaler(), make_column_selector(dtype_include="number")),       # every number column
            ("words", OneHotEncoder(drop="first", handle_unknown="ignore"), make_column_selector(dtype_exclude="number")),
        ])),
        ("logit", LogisticRegression(max_iter=5000)),
    ])
    version.fit(complete.loc[X_train.index, columns], y_train)
    chance = version.predict_proba(two[columns])[:, 1]
    rows[name] = {"test accuracy": version.score(test[columns], y_test),
                  "mean chance, Black": chance[black].mean(), "mean chance, White": chance[white].mean()}
rows["the lenders (share denied)"] = {"mean chance, Black": two.loc[black, "lender_denied"].mean(),
                                      "mean chance, White": two.loc[white, "lender_denied"].mean()}

slopes = pd.Series(version[-1].coef_[0], index=version[0].get_feature_names_out())     # the race version's slopes
odds = np.exp(slopes["words__race_Black or African American"] - slopes["words__race_White"])
print(f"race version: same loan columns, odds of denial × {odds:.1f} for a Black applicant vs a White one")
pd.DataFrame(rows).T.round(3)

<details><summary>Answer</summary>

* **The same accuracy** for all three, 88.2–88.3%. Accuracy can't see any of this
* **With race,** the model copies the lenders' gap: 32.6% against 13.7% (the lenders: 30.7% against 12.9%). With the same loan columns, the odds of denial are **2.3 times** higher for a Black applicant. That is the part of the lenders' gap the six loan columns don't explain
* **Without race** (Week 11), the gap shrinks, 22.6% against 15.0%, but it doesn't vanish: the loan columns themselves differ by group
* **With the neighborhood** instead, part of the gap comes back: 25.5% against 14.6%. The minority share stands in for race. A stand-in like this is a **proxy**, and judging people by their neighborhood is how redlining worked. In Fourcade & Healy's words: "Group-level differences that the law kicks out the door come back in through the window" (ch. 7)
</details>

> "We might turn the analytical logic on its head and argue that an observed gap between groups—an unexplained statistical 'residual'—is in fact a measure of advantages or disadvantages that have been produced and reproduced over the very *longue durée*."
> — Fourcade & Healy, *The Ordinal Society*, ch. 7

> 💬 **Question:** The lenders denied Black applicants more often than the loan columns explain. What could that residual be?

<details><summary>Answer</summary>

* This file can't tell. Lenders saw **credit scores** and credit histories, and HMDA doesn't publish them. In 1992, researchers at the Boston Fed added credit histories from lenders' files, and minority applicants were still denied more often than similar white applicants: about 17% against 11%
* Fourcade & Healy's point: even a credit score, built with no demographic column, carries the past. They quote Barbara Underwood: "a nonracial predictor that correlates highly with race has some of the effects of selection based explicitly on race"
* And the model learns from the lenders' decisions. A model that copies a biased answer key well is biased too: **question the label** (Week 12)
</details>

---
## 4 · Fair by Which Measure? (SLO 14A) · ~10 min

Back to Week 11's model, with no race. Could the lender close a gap by moving the threshold for one group?

> 📄 **Handout 2 — Move one line.** On the Black strip, draw a second line so the model denies 3 of 12 Black applicants, as many as White ones. Count again. 🗣️ What got better, and for whom? What got worse?

<details><summary>Answer</summary>

* The new line sits at about **0.30**: the model denies 3 of 12 in both groups
* Wrongly denied: **1 of 8** Black applicants (was 3 of 8). Now the approved White applicants (2 of 10) are turned away more often
* Missed: still 2 of 4. With so few applicants, the counts move in steps. The full test set shows it better:
</details>

In [ ]:
white_at_02 = measures[(measures["race"] == "White") & (measures["threshold"] == 0.2)].iloc[0]
black_by_threshold = measures[measures["race"] == "Black or African American"].set_index("threshold")
colors = {"model denies": "#2a78d6", "wrongly denied": "#eb6834", "missed": "#1a9850"}
fig = px.line(black_by_threshold, y=list(colors), color_discrete_map=colors,
              labels={"threshold": "threshold for Black applicants", "value": "share", "variable": ""},
              title="Black Applicants by Their Threshold (Dashed: White Applicants at 0.2)")
for measure, color in colors.items():
    fig.add_hline(y=white_at_02[measure], line_dash="dash", line_color=color)
fig.show()

Where each solid line crosses its dashed line, Black and White applicants are equal on that measure. For each measure, the Black threshold that matches White applicants:

In [ ]:
matches = {}
for measure in ["model denies", "wrongly denied", "missed", "precision", "accuracy"]:
    t = (black_by_threshold[measure] - white_at_02[measure]).abs().idxmin()     # the closest match
    matches[f"equal {measure}"] = {"Black threshold": t, **black_by_threshold.loc[t].drop("race")}
pd.DataFrame(matches).T.astype(float).round(3)

<details><summary>Answer</summary>

* **Every measure asks for its own threshold:** equal denials at 0.32, equal *wrongly denied* at 0.27, equal *missed* at 0.20, equal accuracy at 0.41, and equal precision at 0.05, which denies 85% of Black applicants
* **Match one, and the others split apart.** At 0.32 both groups are denied equally, but the model now misses 47% of the Black applicants the lenders denied (White: 34%), and wrongly denies 13% of the approved ones (White: 20%)
* **It isn't bad luck.** The lenders denied 31% of Black and 13% of White applicants. When the groups' **base rates** differ, an imperfect model can't have equal precision and equal mistakes at once (Kleinberg, Mullainathan & Raghavan, 2016; Chouldechova, 2017). Fairness criteria compete. Someone has to choose
* **And a lender may not choose this way.** A threshold by race is a decision by race, **disparate treatment**, illegal in lending. Week 11's single threshold is what the law allows
</details>

Which criterion, then? It depends on what you think fairness asks for. Three views (Barocas, Hardt & Narayanan, *Fairness and Machine Learning*):

| view | fairness asks… | fits | trouble |
|---|---|---|---|
| **narrow** | treat people with the same qualifications, as measured today, the same | unawareness, predictive parity | today's measures carry yesterday's injustice |
| **middle** | don't hold against people the gaps that past injustice created | equal opportunity | costs the decision-maker some accuracy; hard to check |
| **broad** | people of equal ability and effort should end up equally well off | demographic parity | far beyond one lender's decisions; slow |

> 🗣️ **Pair Talk:** Which view should a lender in Kent County follow? And the government that writes the fair-lending laws?

<details><summary>Answer</summary>

* No calculation settles it: it's a question about justice, and the views disagree about what a past injustice still owes
* The law asks lenders for something like the narrow view (the same rule, no race), plus a check from the middle view: a neutral rule that turns away one group more often must be justified by business need, with no less harmful alternative
* The broad view is for public policy: down-payment help, investment in the neighborhoods that were redlined
</details>

---
## 5 · A Toolbox for Fairness: Which Criterion, When? (SLO 14A) · ~7 min

**The toolbox:** every check and criterion from today, why it matters, when to reach for it, and how to compute it. In the code column, `denies` is the model's decision, `approved` and `denied` are the true labels, and `group` is the protected column.

| | option | what is equal across groups | why it matters | when to use it | how, in code |
|---|---|---|---|---|---|
| | ***Always*** | | | | |
| 1 | **Audit by group** | — | one accuracy hides how each group is treated | every model that decides about people, on the test set | `groupby(group)` on each measure; `fairlearn.metrics.MetricFrame` |
| 2 | **Question the label** | — | a model learns its answer key, unfairness included | before modeling, when the label is a past human decision | — |
| | ***What to make equal*** | | | | |
| 3 | **Unawareness** | the inputs: no protected column | no decision *by* race, sex or age | required in U.S. lending. Never enough alone: test for proxies (Part 3) | `X.drop(columns=["race", "sex", "age"])` |
| 4 | **Demographic parity** | the share denied (or selected) | outcomes shouldn't depend on the group | the broad view, or a label you can't trust. Ignores real differences between groups | `denies.groupby(group).mean()` |
| 5 | **Equal opportunity** | the share *wrongly denied*, among those who'd qualify | the harm falls on the qualified applicant | the middle view, when turning people away is the harm that matters | `denies[approved].groupby(group).mean()` |
| 6 | **Equalized odds** | both mistakes: *wrongly denied* and *missed* | each side's risk is shared equally | when both mistakes matter. Costs the most accuracy | the shares in 5, plus `(~denies)[denied].groupby(group).mean()` |
| 7 | **Predictive parity** | precision: of the denials, the share that hold up | a denial means the same in every group | the narrow view; the decision-maker's view | `denied[denies].groupby(group).mean()` |
| 8 | **Calibration** | a chance of 30% means 30%, in every group | people read the score itself | risk scores that people read (doctors, judges) | `denied.groupby([group, pd.cut(chance, 5)]).mean()` |
| 9 | **Accuracy equality** | accuracy | the model is as reliable for everyone | rarely alone: it hides opposite mistakes | `(denies == denied).groupby(group).mean()` |
| | ***Beyond groups*** | | | | |
| 10 | **Individual fairness** | similar people get similar decisions | groups are coarse; people aren't averages | when you can say what makes two applicants "similar" | — |
| 11 | **Counterfactual fairness** | the decision, had only the person's race been different | asks about this person, not a group | needs a causal model of how race shaped the other columns | — |

When the groups' base rates differ, 4–8 can't all hold at once (Part 4).

**Fixes, by when they act** (fairlearn: `pip install fairlearn`):

| when | fix | in code | catch |
|---|---|---|---|
| before training | a better label (did the borrower repay?); more data on small groups; test each column for proxies | can the columns predict race? `LogisticRegression().fit(X, race)` | only approved applicants have a repayment record |
| while training | reweigh the rows, or train under a fairness constraint | `fairlearn.reductions.ExponentiatedGradient(model, constraints=EqualizedOdds())` | usually costs some accuracy |
| after training | a threshold per group | `fairlearn.postprocessing.ThresholdOptimizer` | in lending, a threshold by race is illegal |

**How to choose:** 1 and 2 always. Then ask who pays for each mistake (Week 11): the applicant turned away, the lender, or the public? Choose the criterion that matches that harm and your view of fairness, say so out loud, and report the other measures next to it. Then check what the law allows.

> 📄 **Handout 3 — Pick the criterion.** Write the letter that fits each situation best. 🗣️ A pretrial risk tool, and both sides were right?

<details><summary>Answer</summary>

1. **A.** Unawareness, by law. And ZIP code is on the list because it would be a proxy
2. **E.** Calibration: a 30% must mean 30% for every patient
3. **F.** The label is the managers' past decisions. A model that copies them well copies their bias
4. **C.** Equal opportunity: the qualified are turned away more often in one group
5. **B.** Demographic parity: the same share of winners as of applicants

* **The pretrial tool (COMPAS, 2016):** reporters found that Black defendants who were *not* re-arrested had been flagged high-risk 45% of the time, white ones 23%: unequal *wrongly flagged* (equal opportunity ✗). Its maker answered that each score meant the same chance of re-arrest in both groups (calibration ✓). Both were right. Re-arrest rates differed between the groups, so the tool couldn't have both (Chouldechova, 2017): Part 4, in a courtroom
</details>

---
### Wrap-up

1. **Audit by group:** one accuracy can hide very different treatment (Part 2)
2. **Leaving race out isn't neutral:** the model's columns differ by group, and a proxy like the neighborhood brings race back. Putting race in copies the lenders' gap
3. **Fairness criteria compete:** when base rates differ, equal denials, equal mistakes and equal precision can't all hold. Choosing among them is a question of justice, not of math 📌
4. **The label is a past human decision:** a model that copies it well copies its unfairness too
5. Part 5's table is the checklist for any model that decides about people

**Wednesday:** a model more accurate than this one, and the applicant's right to know *why*.